In [1]:
!pip install yfinance
!pip install bs4
!pip install nbformat
!pip install --upgrade plotly

     ---------------------------------------- 0.0/949.0 kB ? eta -:--:--
     ---------------------------------------- 0.0/949.0 kB ? eta -:--:--
     ---------------------------------------- 0.0/949.0 kB ? eta -:--:--
     ---------------------------------------- 0.0/949.0 kB ? eta -:--:--
     ---------------------------------------- 0.0/949.0 kB ? eta -:--:--
     ---------------------------------------- 0.0/949.0 kB ? eta -:--:--
     ---------------------------------------- 0.0/949.0 kB ? eta -:--:--
     ---------------------------------------- 0.0/949.0 kB ? eta -:--:--
     ---------------------------------------- 0.0/949.0 kB ? eta -:--:--
     ---------------------------------------- 0.0/949.0 kB ? eta -:--:--
     ---------------------------------------- 0.0/949.0 kB ? eta -:--:--
     ----------- ---------------------------- 262.1/949.0 kB ? eta -:--:--
     ----------- ---------------------------- 262.1/949.0 kB ? eta -:--:--
     ----------- ---------------------------- 2

In [2]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [4]:
import plotly.io as pio
pio.renderers.default = "iframe"
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [33]:
def make_graph(stock_data, revenue_data, stock):
    # Filter data up to June 2021
    stock_data_specific = stock_data[stock_data.Date <= '2021-06-30']
    revenue_data_specific = revenue_data[revenue_data.Date <= '2021-06-30']
    
    # Ensure Revenue column is numeric
    #revenue_data_specific['Revenue'] = pd.to_numeric(revenue_data_specific['Revenue'], errors='coerce')
    revenue_data_specific = revenue_data[revenue_data.Date <= '2021-06-30'].copy()
    revenue_data_specific = revenue_data_specific.dropna(subset=['Revenue'])

    # Create 2-row subplot
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                        vertical_spacing=0.1, subplot_titles=(f"{stock} Share Price", f"{stock} Revenue"), 
                        row_width=[0.3, 0.7])

    # Add stock price trace
    fig.add_trace(go.Scatter(
        x=pd.to_datetime(stock_data_specific['Date']), 
        y=stock_data_specific['Close'].astype(float), 
        name="Share Price"
    ), row=1, col=1)

    # Add revenue trace
    fig.add_trace(go.Scatter(
        x=pd.to_datetime(revenue_data_specific['Date']), 
        y=revenue_data_specific['Revenue'], 
        name="Revenue"
    ), row=2, col=1)

    # Update layout and axes
    fig.update_layout(
        title_text=f"{stock} Stock Price and Revenue (Up to June 2021)",
        height=600
    )
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Stock Price (USD)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue (USD)", row=2, col=1)

    fig.show()

In [10]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period = "max")
tesla_data.reset_index(inplace = True)   


In [12]:
Revenue_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
Revenue_data = requests.get(url)

In [13]:
Revenue_soup = BeautifulSoup(html_data.content,"html.parser")


In [17]:
tesla_revenue = pd.DataFrame(columns=["Date", "Revenue"])
tesla2table = Revenue_soup.find_all("tbody")[1]

for row in tesla2table.find_all("tr"):
    col = row.find_all("td")
    if col:
        date = pd.to_datetime(col[0].text)
        revenue = col[1].text
        new_row = pd.DataFrame([{"Date": date, "Revenue": revenue}])
        tesla_revenue = pd.concat([tesla_revenue, new_row], ignore_index=True)
#tesla_revenue


In [29]:
tesla_revenue["Revenue"] = tesla_revenue['Revenue'].str.replace(r',|\$', "", regex=True)


In [34]:
make_graph(tesla_data, tesla_revenue, 'Tesla')